# 환경 설정

In [1]:
# 필요한 라이브러리 / 모듈 / 클래스 / 함수 임포트
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 데이터 불러오기

In [2]:
# 파일 경로 설정
file_path='./데이터/titanic_cleaned.csv'

# DataFrame 생성
df = pd.read_csv(file_path)

# 결과 확인
display(df)

,Survived,Pclass,SibSp,Parch,Fare,Age_step,Gender_female,Gender_male,Embarked_C,Embarked_Q,Embarked_S
0,0,3,1,0,7.2500,2,0,1,0,0,1
1,1,1,1,0,66.3438,3,1,0,1,0,0
2,1,3,0,0,7.9250,2,1,0,0,0,1
3,1,1,1,0,53.1000,3,1,0,0,0,1
4,0,3,0,0,8.0500,3,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
1303,0,3,0,0,8.0500,2,0,1,0,0,1
1304,1,1,0,0,66.3438,3,1,0,1,0,0
1305,0,3,0,0,7.2500,3,0,1,0,0,1
1306,0,3,0,0,8.0500,2,0,1,0,0,1


# 학습용과 평가용 데이터 생성

In [3]:
# 80:20의 비율로 학습용 데이터와 평가용 데이터 생성

## X_data 생성: 정답 컬럼 제거
X_data = df.drop(columns=['Survived'])
display(X_data)

print('-'*80)

## y_data 생성: 정답 컬럼 인덱싱
y_data = df['Survived']
print(y_data)

print('-'*80)

## train_test_split() 함수 호출 -> 무작위성 통제 -> random_state, strafity
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=0,
    stratify=y_data
)

,Pclass,SibSp,Parch,Fare,Age_step,Gender_female,Gender_male,Embarked_C,Embarked_Q,Embarked_S
0,3,1,0,7.2500,2,0,1,0,0,1
1,1,1,0,66.3438,3,1,0,1,0,0
2,3,0,0,7.9250,2,1,0,0,0,1
3,1,1,0,53.1000,3,1,0,0,0,1
4,3,0,0,8.0500,3,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...
1303,3,0,0,8.0500,2,0,1,0,0,1
1304,1,0,0,66.3438,3,1,0,1,0,0
1305,3,0,0,7.2500,3,0,1,0,0,1
1306,3,0,0,8.0500,2,0,1,0,0,1


--------------------------------------------------------------------------------
0       0
1       1
2       1
3       1
4       0
       ..
1303    0
1304    1
1305    0
1306    0
1307    0
Name: Survived, Length: 1308, dtype: int64
--------------------------------------------------------------------------------


In [4]:
# 학습용 데이터 확인: 인덱스 확인
print(f'X_train의 인덱스: {X_train.index}')
print('-'*80)
print(f'y_train의 인덱스: {y_train.index}')

print('-'*80)

# 정답의 항목별 비율 확인
ratio = y_train.value_counts(normalize=True)
print(f'학습용 데이터의 정답별 비율: \n{ratio}')

X_train의 인덱스: Index([ 772,  180,  188, 1128,  194, 1012,  212,  788,  229,   12,
       ...
        377, 1016,  934,   58,  122,  767,  279,  683, 1029,  403],
      dtype='int64', length=1046)
--------------------------------------------------------------------------------
y_train의 인덱스: Index([ 772,  180,  188, 1128,  194, 1012,  212,  788,  229,   12,
       ...
        377, 1016,  934,   58,  122,  767,  279,  683, 1029,  403],
      dtype='int64', length=1046)
--------------------------------------------------------------------------------
학습용 데이터의 정답별 비율: 
Survived
0    0.622371
1    0.377629
Name: proportion, dtype: float64


# 전처리: 표준화

In [5]:
# 표준화 도구 생성
scaler = StandardScaler()

In [6]:
# 학습용과 평가용 데이터 변환: 학습용 기준(평균, 표준편차)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 변환의 결과 확인: pandas 자료형(DataFrame, Series) -> 넘파이 배열
print(X_train_scaled)
print('-'*80)
print(X_test_scaled)

[[-0.37243102 -0.4879889  -0.44752246 ... -0.49970122 -0.31606146
   0.64008086]
 [ 0.82622388  7.23127597  1.95920501 ... -0.49970122 -0.31606146
   0.64008086]
 [ 0.82622388  0.47691921  0.75584127 ... -0.49970122  3.16394158
  -1.56230262]
 ...
 [ 0.82622388  4.33655164  1.95920501 ... -0.49970122 -0.31606146
   0.64008086]
 [ 0.82622388 -0.4879889  -0.44752246 ... -0.49970122 -0.31606146
   0.64008086]
 [ 0.82622388  0.47691921 -0.44752246 ... -0.49970122 -0.31606146
   0.64008086]]
--------------------------------------------------------------------------------
[[ 0.82622388 -0.4879889   1.95920501 ... -0.49970122 -0.31606146
   0.64008086]
 [ 0.82622388 -0.4879889  -0.44752246 ... -0.49970122 -0.31606146
   0.64008086]
 [-0.37243102 -0.4879889  -0.44752246 ... -0.49970122 -0.31606146
   0.64008086]
 ...
 [-1.57108591 -0.4879889   0.75584127 ...  2.00119581 -0.31606146
  -1.56230262]
 [ 0.82622388 -0.4879889  -0.44752246 ... -0.49970122 -0.31606146
   0.64008086]
 [-0.37243102 -0.

In [7]:
# Pytorch Tensor 변환 및 디바이스로 이동
"""
# 자동 미분, 역전파, GPU 상의 연산 기능을 추가하기 위해서
numpy 배열을 Pytorch Tensor로 변환
"""
X_train_tensor = torch.FloatTensor(X_train_scaled)
# print(X_train_tensor)
X_test_tensor = torch.FloatTensor(X_test_scaled)
# print(X_test_tensor
y_train_tensor = torch.FloatTensor(y_train.values)

In [8]:
# pytorch용 표준 데이터셋 생성
"""
1. 학습용 문제(X_train_tensor)와 정답(y_train_tensor)을 하나의 쌍으로 묶어준다.
2. 이유: 학습용 데이터를 미니배치(N) 단위로 샘플링해야 하므로, 서로의 인덱스를
동일하게 만들기 위해서 문제(X_train_tensor)와 정답(y_train_tensor)을 하나의 쌍으로
묶어서 관리 -> TensorDataset 클래스
"""
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

In [9]:
# DataLoader 생성
"""
1. 딥러닝의 학습: 미니 배치 단위
2. 데이터를 미니 배치 단위로 샘플링해서 모델에게 공급해주는 도구
"""
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# MLP 분류 모델 생성

In [10]:
# 입력 특성(컬럼) 수 추출
input_dim = X_train_tensor.shape[1]
print(f'입력 데이터의 특성의 수: {input_dim}')

print('-'*80)

# nn.Sequential 클래수 사용: 입력층 -> 은닉층1 -> 은닉층2 -> 출력층으로 구성
binary_model = nn.Sequential(
    nn.Linear(input_dim, 32), # [입력층 -> 은닉층1]: 입력 10개, 인공 뉴런 32개로 구성
    nn.ReLU(),                # [활성화 함수]: 비선형 연산(음수 -> 0)
    nn.Linear(32, 16),        # [은닉층1 -> 은닉층2]: 입력 32, 인공 뉴런 16개로 구성
    nn.ReLU(),                # [활성화 함수]: 비선형 연산(음수 -> 0)
    nn.Linear(16, 1),         # [은닉층2 -> 출력층]: 입력 16, 출력 뉴런 1개로 구성
)

print(f'생성된 모델 아키텍처: \n{binary_model}')

입력 데이터의 특성의 수: 10
--------------------------------------------------------------------------------
생성된 모델 아키텍처: 
Sequential(
  (0): Linear(in_features=10, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=1, bias=True)
)


# 손실 함수 설정

In [11]:
# 손실 함수: 시그모이드와 이진 교차 엔트로피(BCE)를 결합한 BCEWithLogitsLoss 클래스 적용
criterion = nn.BCEWithLogitsLoss()

# 최적화 알고리즘 설정

In [12]:
# 최적화: Adam 적용(SGD 개선)
"""
1. 학습의 대상 -> parameter -> 뉴런의 가중치와 편향(W, b)
2. Adam() 클래스 첫 번째 hyperparamter -> params = model.parameters()
3. Adam() 클래스 두 번째 hyperparamter -> lr(학습) = 0.001(기본값)
"""
optimizer = optim.Adam(params=binary_model.parameters(), lr=0.005)

# 모델 학습

In [13]:
# 학습 횟수 설정
epochs = 50

# 모델을 훈련 모드로 설정
binary_model.train()

# for 반복문 이용 학습 진행
for epoch in range(epochs):

    ## 손실값 초기화
    epoch_loss = 0.0

    for batch_x, batch_y in train_loader:
        # 1) 기울기 초기화(이전 학습의 결과물 삭제)
        optimizer.zero_grad()

        # 2) 순전파 연산 (출력층 뉴런 입력의 총합)
        outputs = binary_model(batch_x)

        # 3) 손실 계산 (batch_y를 (N, 1) 모양으로 맞춰줌)
        loss = criterion(outputs, batch_y.unsqueeze(1))

        # 4) 오차역전파 (가중치별 기울기 산출)
        loss.backward()

        # 5) 가중치 매개변수 갱신
        optimizer.step()

        # 손실값 업데이트: pytorch Tensor 자료형 파이썬 숫자형으로 변환
        epoch_loss += loss.item()

    # 10 에포크마다 진행 상황 출력
    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f'Epoch [{epoch+1}/{epochs}], Train BCE Loss: {avg_loss:.4f}')

Epoch [1/50], Train BCE Loss: 0.5452
Epoch [10/50], Train BCE Loss: 0.3494
Epoch [20/50], Train BCE Loss: 0.3382
Epoch [30/50], Train BCE Loss: 0.3281
Epoch [40/50], Train BCE Loss: 0.3184
Epoch [50/50], Train BCE Loss: 0.3105
